# PS1 3-day GATE — Improved Baseline (models only)

Focused modeling notebook: **no Optuna, no MLflow, no SageMaker deploy**.

**Silver join notes (catalog):**
| Table | Join | GATE note |
|---|---|---|
| `usage_lifecycle_daily`, `metric_daily`, `maintenance_ledger` | `DEVICE_KEY` + `event_date` | ~95% coverage |
| `station_network_daily` | `FACILITY_ID` + `GATE` + `event_date` | **sparse** (~2% raw rows — failure days only) |
| `read_tap_daily` | — | **skipped for GATE** (EDW read txns; use Gold `tap_reject_rate_pct`) |
| `device_outage`, `device_ps2_chains` | `DEVICE_KEY` + `event_date` | subsystem / cascade |

**Modeling choices (v2):**
- SPW sweep on **pre-oversample** train → use `spw_train` for boosters (not post-ROS ratio)
- Light oversampling (5% pos) + val AP-weighted ensemble
- **Deployment candidate**: `deploy_ap_isotonic_ensemble` (LGB/XGB/CB + val AP weights + guarded isotonic)
- Guarded isotonic (skip if val AP drops)
- Gold tap columns protected; MI top 100; shift(1) leakage discipline

In [ ]:
# CELL 1 — Install
import subprocess, sys
pkgs = ["s3fs", "pyarrow", "scikit-learn>=1.4", "imbalanced-learn>=0.12",
        "xgboost>=2.0", "lightgbm>=4.3", "catboost>=1.2", "pandas>=2.0",
        "matplotlib", "seaborn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Packages OK.")

In [ ]:
# CELL 2 — Imports, config, join helpers
import os, warnings
import decimal as _decimal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    f1_score, precision_score, recall_score, roc_curve, confusion_matrix,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier
from sklearn.isotonic import IsotonicRegression
from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_S3     = f"s3://{GOLD_BUCKET}/chicago/gold/device_ps1_daily"
SILVER      = f"s3://{GOLD_BUCKET}/chicago/silver"
REGION      = "us-east-1"
S3_OPTS     = {"client_kwargs": {"region_name": REGION}}

TARGET           = "will_fail_3d"
VAL_START        = pd.Timestamp("2025-10-01")
TEST_START       = pd.Timestamp("2026-01-01")
EMBARGO_DAYS     = 14
RECALL_FLOOR     = 0.70
NULL_DROP_THRESH = 0.90
ZERO_DROP_THRESH = 0.95   # drop near-constant cols on train (>95% zeros)
USE_MI_PRUNING   = True
MI_TOP_N         = 100
LEAKAGE_AUC_TH   = 0.95
SPW_CANDIDATES    = [10, 15, 20, 25, 30]
OVERSAMPLE_RATIO  = 0.05
MIN_BOOSTER_VAL_AP = 0.02          # drop collapsed boosters from ensemble
USE_ISOTONIC      = True           # guarded: only if val AP improves
TAP_PROTECT_PAT   = ("tap_reject", "tap_volume", "peak_hour_tap", "tap_event", "reject_rate")
OUT_DIR           = "ps1_gate_improved_outputs"
os.makedirs(OUT_DIR, exist_ok=True)


def _cast_decimals(frame):
    dec_cols = [c for c in frame.columns if frame[c].dtype == object
                and len(frame[c].dropna()) > 0
                and isinstance(frame[c].dropna().iloc[0], _decimal.Decimal)]
    if dec_cols:
        frame[dec_cols] = frame[dec_cols].apply(pd.to_numeric, errors="coerce")
    return frame


def _norm_facility(s):
    """Align FACILITY_ID types between gold (decimal) and silver (numeric/string)."""
    return pd.to_numeric(s, errors="coerce")


def _report_match(df, col, label):
    if col not in df.columns:
        print(f"  [{label}] {col}: missing")
        return 0.0
    nn = df[col].notna().mean()
    nz = (df[col].fillna(0) != 0).mean()
    print(f"  [{label}] {col}: notna={nn:.1%}  non-zero={nz:.1%}")
    return nn


def _report_signal(df, col, label):
    """Alias — reports both coverage and non-zero rate (meaningful for sparse tables)."""
    return _report_match(df, col, label)


def _prior_roll(g, col, window, how="mean"):
    """shift(1) then rolling — catalog leakage discipline."""
    if how == "mean":
        return g[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
    if how == "sum":
        return g[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).sum())
    if how == "max":
        return g[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).max())
    raise ValueError(how)

print(f"Gold: {GOLD_S3}")

In [ ]:
# CELL 3 — Load Gold, GATE filter, column aliases
df = pd.read_parquet(GOLD_S3, storage_options=S3_OPTS)
if "transit_day" in df.columns and "event_date" not in df.columns:
    df = df.rename(columns={"transit_day": "event_date"})
df["event_date"] = pd.to_datetime(df["event_date"])
df = df[df["mars_device_category"] == "GATE"].copy()
df = _cast_decimals(df)
df = df.sort_values(["DEVICE_ID", "event_date"]).reset_index(drop=True)

if "event_count" in df.columns and "daily_event_count" not in df.columns:
    df = df.rename(columns={"event_count": "daily_event_count"})
if "oos_event_count" in df.columns and "oos_flag" not in df.columns:
    df["oos_flag"] = (df["oos_event_count"] > 0).astype(int)

print(f"GATE rows: {len(df):,}  devices: {df['DEVICE_ID'].nunique():,}")
print(f"DEVICE_KEY coverage: {df['DEVICE_KEY'].notna().mean():.1%}")
print(f"Date range: {df['event_date'].min().date()} -> {df['event_date'].max().date()}")
print(f"Target rate: {df[TARGET].mean():.2%}")
_tap_gold = [c for c in df.columns if any(p in c.lower() for p in TAP_PROTECT_PAT)]
print(f"Gold tap columns ({len(_tap_gold)}): {_tap_gold[:8]}{'...' if len(_tap_gold)>8 else ''}")
for _c in _tap_gold[:6]:
    _report_signal(df, _c, f"GOLD_TAP {_c}")

In [ ]:
# CELL 4 — Silver enrichments (schema-correct joins)
# Each block is independent; failures are logged and skipped.

def _g():
    return df.groupby("DEVICE_ID", sort=False)

print("Silver enrichment join audit:")

# --- maintenance_ledger: DEVICE_KEY + event_date ---
try:
    maint = pd.read_parquet(f"{SILVER}/maintenance_ledger", storage_options=S3_OPTS,
                            columns=["DEVICE_KEY", "event_dtm", "ledger_type", "mars_device_category"])
    maint = maint[(maint["mars_device_category"] == "GATE") & (maint["ledger_type"] != "REPAIR_EPISODE")].copy()
    maint["event_date"] = pd.to_datetime(maint["event_dtm"]).dt.normalize()
    md = maint.groupby(["DEVICE_KEY", "event_date"]).size().reset_index(name="_maint_flag")
    md["_maint_flag"] = 1
    n0 = len(df)
    df = df.merge(md, on=["DEVICE_KEY", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df["_maint_flag"] = df["_maint_flag"].fillna(0).astype(int)
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    df["_maint_lag"] = g["_maint_flag"].shift(1).fillna(0)
    df["maint_visit_days_7d"]  = g["_maint_lag"].transform(lambda x: x.rolling(7, min_periods=1).sum())
    df["maint_visit_days_30d"] = g["_maint_lag"].transform(lambda x: x.rolling(30, min_periods=1).sum())
    df["_last_maint"] = df["event_date"].where(df["_maint_lag"] == 1)
    df["_last_maint"] = g["_last_maint"].ffill()
    df["days_since_maintenance"] = (df["event_date"] - df["_last_maint"]).dt.days.fillna(999).clip(upper=999)
    df = df.drop(columns=["_maint_flag", "_maint_lag", "_last_maint"])
    _report_match(df, "days_since_maintenance", "MAINT")
except Exception as e:
    print(f"  [MAINT] skip: {e}")

# --- device_ps2_chains (gold): DEVICE_KEY + event_date ---
try:
    ps2 = pd.read_parquet(f"s3://{GOLD_BUCKET}/chicago/gold/device_ps2_chains", storage_options=S3_OPTS,
                          columns=["DEVICE_KEY", "transit_day", "is_coordinated_station_failure"])
    ps2["event_date"] = pd.to_datetime(ps2["transit_day"])
    p2 = ps2.groupby(["DEVICE_KEY", "event_date"]).agg(
        _coord=("is_coordinated_station_failure", "max")).reset_index()
    p2["_cascade_flag"] = 1
    p2["_coord_flag"] = p2["_coord"].astype(int)
    n0 = len(df)
    df = df.merge(p2[["DEVICE_KEY", "event_date", "_cascade_flag", "_coord_flag"]],
                  on=["DEVICE_KEY", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    for c in ["_cascade_flag", "_coord_flag"]:
        df[c] = df[c].fillna(0).astype(int)
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    df["_casc_lag"] = g["_cascade_flag"].shift(1).fillna(0)
    df["_coord_lag"] = g["_coord_flag"].shift(1).fillna(0)
    df["station_cascade_days_30d"] = g["_casc_lag"].transform(lambda x: x.rolling(30, min_periods=1).sum())
    df["station_coordinated_fail_days_30d"] = g["_coord_lag"].transform(lambda x: x.rolling(30, min_periods=1).sum())
    df["_last_casc"] = df["event_date"].where(df["_casc_lag"] == 1)
    df["_last_casc"] = g["_last_casc"].ffill()
    df["days_since_station_cascade"] = (df["event_date"] - df["_last_casc"]).dt.days.fillna(999).clip(upper=999)
    df = df.drop(columns=["_cascade_flag", "_coord_flag", "_casc_lag", "_coord_lag", "_last_casc"])
    _report_match(df, "days_since_station_cascade", "PS2")
except Exception as e:
    print(f"  [PS2] skip: {e}")

# --- device_outage subsystems: DEVICE_KEY + event_date ---
try:
    _SUBSYSTEMS = ["SYSTEM", "DOPP", "COMMS", "OTHER"]
    _out = pd.read_parquet(f"{SILVER}/device_outage", storage_options=S3_OPTS,
                           columns=["DEVICE_KEY", "transit_day", "mars_device_category", "duration_min",
                                    "is_chargeable", "failure_level", "component_subsystem"])
    _out = _out[(_out["mars_device_category"] == "GATE") & (_out["duration_min"] > 0)
                & (_out["is_chargeable"] == True)
                & (_out["failure_level"].isin([2, 3, 4, 5]))].copy()
    _out["event_date"] = pd.to_datetime(_out["transit_day"])
    _known = set(_SUBSYSTEMS[:-1])
    _out["_sub"] = np.where(_out["component_subsystem"].isin(_known), _out["component_subsystem"], "OTHER")
    _out_daily = _out.groupby(["DEVICE_KEY", "event_date", "_sub"]).size().unstack("_sub", fill_value=0).reset_index()
    for _s in _SUBSYSTEMS:
        if _s not in _out_daily.columns:
            _out_daily[_s] = 0
        _out_daily[_s] = (_out_daily[_s] > 0).astype(int)
    n0 = len(df)
    df = df.merge(_out_daily, on=["DEVICE_KEY", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    for _s in _SUBSYSTEMS:
        df[_s] = df[_s].fillna(0).astype(int)
        _lagcol = f"_lag_{_s}"
        df[_lagcol] = g[_s].shift(1).fillna(0)
        df[f"roll_fail_{_s}_30d"] = g[_lagcol].transform(lambda x: x.rolling(30, min_periods=1).sum())
        _lastcol = f"_last_{_s}"
        df[_lastcol] = df["event_date"].where(df[_lagcol] == 1)
        df[_lastcol] = g[_lastcol].ffill()
        df[f"days_since_fail_{_s}"] = (
            (df["event_date"] - df[_lastcol]).dt.days.fillna(999).clip(upper=999))
    _temp = _SUBSYSTEMS + [f"_lag_{s}" for s in _SUBSYSTEMS] + [f"_last_{s}" for s in _SUBSYSTEMS]
    df = df.drop(columns=[c for c in _temp if c in df.columns])
    _report_match(df, "days_since_fail_SYSTEM", "SUBSYS")
except Exception as e:
    print(f"  [SUBSYS] skip: {e}")

# --- read_tap_daily: SKIP for GATE ---
# S22 sources EDW.READ_TRANSACTION (validators/readers). GATE uses tap_event_daily in Gold
# (tap_reject_rate_pct, peak_hour_tap_count, etc.) — ~27% GATE coverage vs ~1% read_tap.
print("  [READ_TAP] skipped for GATE — use Gold tap_event_daily columns (tap_reject_rate_pct, ...)")

# --- station_network_daily: sparse failure-day rows only (S27 design) ---
try:
    st = pd.read_parquet(f"{SILVER}/station_network_daily", storage_options=S3_OPTS,
                         columns=["FACILITY_ID", "device_category", "transit_day",
                                  "devices_failed", "total_failure_events",
                                  "is_coordinated_failure", "is_major_station_event"])
    st = st[st["device_category"] == "GATE"].copy()
    st["event_date"] = pd.to_datetime(st["transit_day"])
    st["FACILITY_ID"] = _norm_facility(st["FACILITY_ID"])
    df["_fac_join"] = _norm_facility(df["FACILITY_ID"])
    for c in ["is_coordinated_failure", "is_major_station_event"]:
        st[c] = st[c].astype(int)
    n0 = len(df)
    df = df.merge(st.drop(columns=["transit_day", "device_category"]).rename(columns={"FACILITY_ID": "_fac_join"}),
                  on=["_fac_join", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    _report_signal(df, "devices_failed", "STN_NET raw")
    for c, how, out in [
        ("devices_failed", "sum", "stn_devices_failed_7d_prior"),
        ("total_failure_events", "sum", "stn_failure_events_7d_prior"),
        ("is_coordinated_failure", "max", "stn_coordinated_7d_prior"),
        ("is_major_station_event", "max", "stn_major_7d_prior"),
    ]:
        df[c] = df[c].fillna(0.0)
        df[out] = _prior_roll(g, c, 7, how=how)
        df = df.drop(columns=[c])
    df = df.drop(columns=["_fac_join"])
    _report_signal(df, "stn_devices_failed_7d_prior", "STN_NET prior7d")
    _report_signal(df, "stn_coordinated_7d_prior", "STN_NET coord prior7d")
except Exception as e:
    print(f"  [STN_NET] skip: {e}")

# --- usage_lifecycle_daily: DEVICE_KEY + transit_day (NOT DEVICE_ID alone) ---
try:
    ul = pd.read_parquet(f"{SILVER}/usage_lifecycle_daily", storage_options=S3_OPTS,
                         columns=["DEVICE_KEY", "transit_day", "mars_device_category",
                                  "m401_daily_txn_count", "tap_count_30d", "failure_count_30d"])
    ul = ul[ul["mars_device_category"] == "GATE"].copy()
    ul["event_date"] = pd.to_datetime(ul["transit_day"])
    ul = ul.drop_duplicates(["DEVICE_KEY", "event_date"])
    n0 = len(df)
    df = df.merge(ul.drop(columns=["transit_day", "mars_device_category"]),
                  on=["DEVICE_KEY", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    _report_signal(df, "m401_daily_txn_count", "USAGE raw")
    df["m401_daily_txn_count"] = df["m401_daily_txn_count"].fillna(0)
    df["usage_tap_count_30d_prior"] = g["m401_daily_txn_count"].transform(
        lambda s: s.shift(1).rolling(30, min_periods=1).sum())
    df["usage_tap_count_lifetime_prior"] = g["m401_daily_txn_count"].transform(
        lambda s: s.shift(1).cumsum())
    # Silver cumulative cols include today — recompute lagged priors only
    if "tap_count_30d" in df.columns:
        df["usage_tap_count_30d_silver_prior"] = g["tap_count_30d"].shift(1)
    if "failure_count_30d" in df.columns:
        df["usage_fail_count_30d_silver_prior"] = g["failure_count_30d"].shift(1)
    df = df.drop(columns=["m401_daily_txn_count", "tap_count_30d", "failure_count_30d"], errors="ignore")
    _report_signal(df, "usage_tap_count_30d_prior", "USAGE prior")
except Exception as e:
    print(f"  [USAGE] skip: {e}")

# --- metric_daily: DEVICE_KEY + transit_day (comms / slow-tap signal) ---
try:
    md = pd.read_parquet(f"{SILVER}/metric_daily", storage_options=S3_OPTS,
                         columns=["DEVICE_KEY", "transit_day", "mars_device_category",
                                  "m401_slow_tap_pct", "m401_z_score_vs_28d",
                                  "comms_total_count", "volume_drop_flag"])
    md = md[md["mars_device_category"] == "GATE"].copy()
    md["event_date"] = pd.to_datetime(md["transit_day"])
    md = md.drop_duplicates(["DEVICE_KEY", "event_date"])
    n0 = len(df)
    df = df.merge(md.drop(columns=["transit_day", "mars_device_category"]),
                  on=["DEVICE_KEY", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    _report_signal(df, "m401_slow_tap_pct", "METRIC_DAILY raw")
    for c in ["m401_slow_tap_pct", "m401_z_score_vs_28d", "comms_total_count"]:
        if c in df.columns:
            df[f"{c}_lag1"] = g[c].shift(1)
    if "volume_drop_flag" in df.columns:
        df["volume_drop_flag_lag1"] = g["volume_drop_flag"].astype(float).shift(1).fillna(0)
    df = df.drop(columns=["m401_slow_tap_pct", "m401_z_score_vs_28d",
                          "comms_total_count", "volume_drop_flag"], errors="ignore")
    _report_signal(df, "m401_slow_tap_pct_lag1", "METRIC_DAILY lag1")
except Exception as e:
    print(f"  [METRIC_DAILY] skip: {e}")

print(f"\nShape after silver enrichments: {df.shape}")
_enrich_cols = [c for c in df.columns if any(x in c for x in (
    "maint_", "station_", "stn_", "roll_fail_", "days_since_fail_",
    "usage_", "m401_", "comms_", "volume_drop"))]
print(f"Enrichment feature candidates: {len(_enrich_cols)}")

In [ ]:
# CELL 5 — Notebook feature blocks A–L (strict shift(1), no same-day leakage)

ID_COLS = ["DEVICE_ID", "event_date", "split", "DEVICE_KEY", "FACILITY_ID",
           "OPERATOR_ID", "BUS_ID", "TRANSIT_ARRAY_ID", "ARRAY_POSITION", "mars_device_category"]
LABEL_LEAK = ["failure_level", "max_failure_level", "is_chargeable", "is_hardware_oos",
              "is_hardware_event", "duration_min", "oos_duration_min",
              "will_fail_7d", "will_fail_14d", TARGET]

g = df.groupby("DEVICE_ID", sort=False)

# Skip notebook rolling cols when Gold already ships the same signal (avoid diluting MI).
_GOLD_SKIP = {
    "oos_flag_roll7": "oos_events_7d",
    "oos_flag_roll30": "oos_events_30d",
    "daily_event_count_roll7": "events_7d",
}
for col, window in [("oos_flag", 7), ("oos_flag", 30), ("daily_event_count", 7)]:
    nb = f"{col}_roll{window}"
    if col in df.columns and nb not in _GOLD_SKIP:
        df[nb] = g[col].transform(lambda x: x.shift(1).rolling(window, min_periods=1).sum())
    elif col in df.columns and _GOLD_SKIP.get(nb) in df.columns:
        print(f"  [FE] skip {nb} — gold has {_GOLD_SKIP[nb]}")

if "daily_event_count" in df.columns:
    df["event_rate_zscore"] = g["daily_event_count"].transform(
        lambda x: (x.shift(1) - x.shift(1).expanding().mean()) / (x.shift(1).expanding().std() + 1e-6))

if "use_revenue_cents" in df.columns:
    df["use_revenue_cents_roll7"] = g["use_revenue_cents"].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).mean())

for col in [c for c in ["oos_flag", "daily_event_count", "use_revenue_cents"] if c in df.columns]:
    for lag in [1, 2, 3]:
        df[f"{col}_lag{lag}"] = g[col].shift(lag)

if "oos_flag" in df.columns:
    for w in [30, 90]:
        df[f"oos_count_{w}d"] = g["oos_flag"].transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).sum())
    def _days_since_oos(x):
        s = x.shift(1); cs = s.cumsum()
        out = cs.groupby(cs).cumcount().astype(float); out[cs == 0] = np.nan; return out
    df["days_since_last_oos"] = g["oos_flag"].transform(_days_since_oos)
    df["oos_lifetime_count"] = g["oos_flag"].transform(lambda x: x.shift(1).cumsum())

df["device_age_days"] = g["event_date"].transform(lambda x: (x - x.min()).dt.days)

if "oos_flag" in df.columns:
    def _consec(x):
        s = x.shift(1).fillna(0); grp = (s != s.shift(1)).cumsum()
        streak = s.groupby(grp).cumcount() + 1; streak[s == 0] = 0.0; return streak.astype(float)
    df["consec_oos_days"] = g["oos_flag"].transform(_consec)
    df["max_streak_30d"] = g["consec_oos_days"].transform(lambda x: x.shift(1).rolling(30, min_periods=1).max())
    r7  = g["oos_flag"].transform(lambda x: x.shift(1).rolling(7, min_periods=1).sum())
    p7  = g["oos_flag"].transform(lambda x: x.shift(8).rolling(7, min_periods=1).sum())
    r30 = g["oos_flag"].transform(lambda x: x.shift(1).rolling(30, min_periods=1).sum())
    p30 = g["oos_flag"].transform(lambda x: x.shift(31).rolling(30, min_periods=1).sum())
    df["oos_rate_accel_7d"]  = (r7 - p7).fillna(0)
    df["oos_rate_accel_30d"] = (r30 - p30).fillna(0)
    df["oos_ewm7"]  = g["oos_flag"].transform(lambda x: x.shift(1).ewm(span=7, min_periods=1).mean())
    df["oos_ewm30"] = g["oos_flag"].transform(lambda x: x.shift(1).ewm(span=30, min_periods=1).mean())
    l1 = g["oos_flag"].shift(1).fillna(0); l2 = g["oos_flag"].shift(2).fillna(0)
    df["_recovery"] = ((l2 == 1) & (l1 == 0)).astype(float)
    df["recovery_count_30d"] = g["_recovery"].transform(lambda x: x.rolling(30, min_periods=1).sum())
    df["recovery_count_90d"] = g["_recovery"].transform(lambda x: x.rolling(90, min_periods=1).sum())
    df = df.drop(columns=["_recovery"])

if "daily_event_count" in df.columns:
    mu  = g["daily_event_count"].transform(lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    std = g["daily_event_count"].transform(lambda x: x.shift(1).rolling(30, min_periods=5).std())
    df["event_count_zscore_30d"] = ((g["daily_event_count"].shift(1) - mu) / (std + 1e-6)).fillna(0)

df["day_of_week"]  = df["event_date"].dt.dayofweek
df["month"]        = df["event_date"].dt.month
df["day_of_month"] = df["event_date"].dt.day
df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)
df["quarter"]      = df["event_date"].dt.quarter

if "FACILITY_ID" in df.columns and "oos_flag" in df.columns:
    df["_oos_lag1"] = g["oos_flag"].shift(1).fillna(0)
    fac_mean = df.groupby(["event_date", "FACILITY_ID"])["_oos_lag1"].transform("mean")
    df["peer_oos_rate_lag1"] = fac_mean
    df["oos_vs_peer_lag1"]   = df["_oos_lag1"] - fac_mean
    df = df.drop(columns=["_oos_lag1"])

if "oos_ewm7" in df.columns:
    df["oos_ewm7_x_age"] = df["oos_ewm7"] * np.log1p(df["device_age_days"])
if "oos_flag" in df.columns and "daily_event_count" in df.columns:
    df["oos_x_event_count"] = g["oos_flag"].shift(1).fillna(0) * g["daily_event_count"].shift(1).fillna(0)

for dc in ["availability_pct_7d", "outage_min_7d"]:
    if dc in df.columns:
        df[dc] = pd.to_numeric(df[dc], errors="coerce").astype(float)

FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + LABEL_LEAK and pd.api.types.is_numeric_dtype(df[c])]
print(f"Candidate features: {len(FEATURE_COLS)}")

In [ ]:
# CELL 6 — Leakage scan (solo-feature AUC on internal temporal holdout)
_scan_cut = df["event_date"].quantile(0.80)
_scan_tr  = df[df["event_date"] <= _scan_cut]
_scan_va  = df[df["event_date"] >  _scan_cut]

def _solo_auc(col):
    a = _scan_tr[col].astype(float); b = _scan_va[col].astype(float)
    med = a.median(); a = a.fillna(med); b = b.fillna(med)
    if a.nunique() < 2 or _scan_va[TARGET].nunique() < 2:
        return np.nan
    auc = roc_auc_score(_scan_va[TARGET], b)
    return max(auc, 1 - auc)

_scan = pd.Series({c: _solo_auc(c) for c in FEATURE_COLS}).sort_values(ascending=False)
_leaky = _scan[_scan > LEAKAGE_AUC_TH].index.tolist()
print("Top solo-AUC features:")
print(_scan.head(12).round(4).to_string())
if _leaky:
    print(f"\nDropping {len(_leaky)} leaky features: {_leaky}")
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in _leaky]
else:
    print(f"\nNo feature exceeded solo-AUC {LEAKAGE_AUC_TH} — keeping {len(FEATURE_COLS)} features.")

In [ ]:
# CELL 7 — Split, dead-feature drop, impute, MI pruning
tr = df[df["event_date"] <  VAL_START - pd.Timedelta(days=EMBARGO_DAYS)]
va = df[(df["event_date"] >= VAL_START) & (df["event_date"] < TEST_START - pd.Timedelta(days=EMBARGO_DAYS))]
te = df[df["event_date"] >= TEST_START]

null_rate = tr[FEATURE_COLS].isnull().mean()
dead = null_rate[null_rate > NULL_DROP_THRESH].index.tolist()
FEATURE_COLS = [c for c in FEATURE_COLS if c not in dead]
print(f"Dropped {len(dead)} dead features (>={NULL_DROP_THRESH:.0%} null on train)")
if dead:
    for c in dead:
        print(f"  {c:45s} null={null_rate[c]:.1%}")

X_tr_raw = tr[FEATURE_COLS].astype(float)
zero_rate = (X_tr_raw.fillna(0) == 0).mean()
sparse = zero_rate[zero_rate > ZERO_DROP_THRESH].index.tolist()
# Keep sparse cols if they are core failure-history signals (often legitimately mostly zero).
_SPARSE_KEEP = {c for c in sparse if any(k in c for k in (
    "roll_fail", "days_since_fail", "days_since_maintenance", "days_since_station",
    "stn_coordinated", "stn_major", "subsystem", "SYSTEM", "DOPP", "COMMS"))}
sparse = [c for c in sparse if c not in _SPARSE_KEEP]
FEATURE_COLS = [c for c in FEATURE_COLS if c not in sparse]
print(f"Dropped {len(sparse)} near-constant features (>{ZERO_DROP_THRESH:.0%} zero on train, excl. failure-history)")
if sparse:
    for c in sparse[:12]:
        print(f"  {c:45s} zero={zero_rate[c]:.1%}")
    if len(sparse) > 12:
        print(f"  ... +{len(sparse)-12} more")

X_tr = tr[FEATURE_COLS].astype(float)
X_va = va[FEATURE_COLS].astype(float)
X_te = te[FEATURE_COLS].astype(float)
y_tr, y_va, y_te = tr[TARGET].values, va[TARGET].values, te[TARGET].values

train_median = X_tr.median(numeric_only=True).values
train_median = np.where(np.isnan(train_median), 0.0, train_median)

def impute(X):
    arr = X.values.copy()
    mask = np.isnan(arr)
    arr[mask] = np.take(train_median, np.where(mask)[1])
    return arr

X_tr = impute(X_tr); X_va = impute(X_va); X_te = impute(X_te)

if USE_MI_PRUNING:
    mi = mutual_info_classif(X_tr, y_tr, random_state=42, discrete_features=False)
    mi_s = pd.Series(mi, index=FEATURE_COLS).sort_values(ascending=False)
    keep = mi_s.head(MI_TOP_N).index.tolist()
    # Force-keep Gold tap_event_daily signal through MI prune
    _tap_keep = [c for c in FEATURE_COLS if any(p in c.lower() for p in TAP_PROTECT_PAT)]
    keep = list(dict.fromkeys(list(keep) + _tap_keep))
    if _tap_keep:
        print(f"  Tap-protected (+{len(_tap_keep)}): {_tap_keep[:6]}{'...' if len(_tap_keep)>6 else ''}")
    print(f"\nMI pruning: {len(FEATURE_COLS)} -> {len(keep)}")
    print(mi_s.head(12).round(4).to_string())
    idx = [FEATURE_COLS.index(c) for c in keep]
    FEATURE_COLS = keep
    X_tr, X_va, X_te = X_tr[:, idx], X_va[:, idx], X_te[:, idx]

spw_raw = round((y_tr == 0).sum() / max((y_tr == 1).sum(), 1), 2)
print(f"\nTrain {X_tr.shape}  pos={(y_tr==1).sum():,} ({(y_tr==1).mean():.2%})")
print(f"Val   {X_va.shape}  pos={(y_va==1).sum():,} ({(y_va==1).mean():.2%})")
print(f"Test  {X_te.shape}  pos={(y_te==1).sum():,} ({(y_te==1).mean():.2%})")
print(f"scale_pos_weight raw={spw_raw}")

# --- SPW sweep (quick LGB, val AP) ---
_spw_scores = {}
for _cand in SPW_CANDIDATES:
    _probe = LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31, max_depth=6,
        scale_pos_weight=_cand, random_state=42, n_jobs=-1, verbose=-1,
    )
    _probe.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc",
               callbacks=[early_stopping(30, verbose=False), log_evaluation(0)])
    _spw_scores[_cand] = average_precision_score(y_va, _probe.predict_proba(X_va)[:, 1])
spw = max(_spw_scores, key=_spw_scores.get)
spw_train = spw   # boosters: always pre-oversample SPW (prod pattern)
print(f"SPW sweep val AP: {_spw_scores}")
print(f"  -> spw_train={spw_train}  val_AP={_spw_scores[spw]:.4f}")

_ros = RandomOverSampler(sampling_strategy=OVERSAMPLE_RATIO, random_state=42)
X_tr_fit, y_tr_fit = _ros.fit_resample(X_tr, y_tr)
spw_post_ros = round((y_tr_fit == 0).sum() / max((y_tr_fit == 1).sum(), 1), 2)
print(f"Oversample to {OVERSAMPLE_RATIO:.0%}: {X_tr.shape[0]:,} -> {X_tr_fit.shape[0]:,} rows")
print(f"  post-ROS pos rate {(y_tr_fit==1).mean():.2%}  spw_post_ros={spw_post_ros} (do NOT use as scale_pos_weight)")

In [ ]:
# CELL 8 — Metrics + ROC / confusion-matrix plots (all splits)

_SPLIT_DATA = {
    "train": lambda: (y_tr, X_tr),
    "val":   lambda: (y_va, X_va),
    "test":  lambda: (y_te, X_te),
}


def _proba_splits(model):
    out = {}
    for split, (y, X) in [(s, _SPLIT_DATA[s]()) for s in ("train", "val", "test")]:
        out[split] = model.predict_proba(X)[:, 1]
    return out


def plot_roc_and_confusion(name, probas, threshold=0.5, op_threshold=None, save=True):
    """ROC + confusion matrices for train / val / test."""
    ys = {s: _SPLIT_DATA[s]()[0] for s in ("train", "val", "test")}
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f"{name} — ROC & confusion matrices", fontsize=14, y=1.01)

    for ax, split in zip(axes[0], ("train", "val", "test")):
        y, p = ys[split], probas[split]
        fpr, tpr, _ = roc_curve(y, p)
        auc = roc_auc_score(y, p)
        ax.plot(fpr, tpr, lw=2, label=f"AUC={auc:.3f}")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.35)
        ax.set_title(f"ROC — {split.upper()}")
        ax.set_xlabel("False positive rate")
        ax.set_ylabel("True positive rate")
        ax.legend(loc="lower right")

    thr_map = {"train": threshold, "val": threshold, "test": op_threshold or threshold}
    for ax, split in zip(axes[1], ("train", "val", "test")):
        y, p = ys[split], probas[split]
        thr = thr_map[split]
        pred = (p >= thr).astype(int)
        cm = confusion_matrix(y, pred)
        sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=ax, cbar=False,
                    xticklabels=["No Fail", "Fail"], yticklabels=["No Fail", "Fail"])
        ax.set_title(f"CM — {split.upper()}  (t={thr:.4f})")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")

    plt.tight_layout()
    if save:
        safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in name)
        path = os.path.join(OUT_DIR, f"{safe}_roc_confusion.png")
        fig.savefig(path, dpi=120, bbox_inches="tight")
        print(f"  [plot] saved -> {path}")
    plt.show()


def eval_split(y_true, proba, name=""):
    pred05 = (proba >= 0.5).astype(int)
    return {"split": name, "auc": roc_auc_score(y_true, proba),
            "ap": average_precision_score(y_true, proba),
            "acc": (pred05 == y_true).mean(),
            "f1": f1_score(y_true, pred05, zero_division=0),
            "prec": precision_score(y_true, pred05, zero_division=0),
            "rec": recall_score(y_true, pred05, zero_division=0)}

def recall_at_k(y_true, proba, k_pct=0.10):
    k = max(1, int(len(y_true) * k_pct))
    top = np.argsort(proba)[-k:]
    return {"prec_at_k": y_true[top].mean(),
            "recall_at_k": y_true[top].sum() / max(y_true.sum(), 1),
            "lift": y_true[top].mean() / max(y_true.mean(), 1e-9)}

def best_recall_constrained_threshold(y_true, proba, floor=RECALL_FLOOR, max_fleet=0.50):
    order = np.argsort(proba)[::-1]; best_t, best_p = 0.0, 0.0
    for i in range(1, len(y_true) + 1):
        if i / len(y_true) > max_fleet: break
        t = proba[order[i - 1]]; pred = (proba >= t).astype(int)
        rec = recall_score(y_true, pred, zero_division=0)
        if rec >= floor:
            prec = precision_score(y_true, pred, zero_division=0)
            if prec > best_p: best_p, best_t = prec, t
    return best_t, best_p

def calibrate_isotonic(raw_va, y_va_ref, raw_te):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(raw_va, y_va_ref)
    cal_va = iso.predict(raw_va)
    cal_te = iso.predict(raw_te)
    if average_precision_score(y_va_ref, cal_va) >= average_precision_score(y_va_ref, raw_va):
        return cal_va, cal_te, iso, True
    return raw_va, raw_te, None, False


def val_ap_weights_named(models_dict, X_va_ref, y_va_ref, min_ap=MIN_BOOSTER_VAL_AP):
    active = {}
    for name, m in models_dict.items():
        ap = average_precision_score(y_va_ref, m.predict_proba(X_va_ref)[:, 1])
        if ap >= min_ap:
            active[name] = (m, ap)
        else:
            print(f"  [ensemble] dropping {name}: val_AP={ap:.4f} < {min_ap}")
    if not active:
        raise RuntimeError("All boosters collapsed — check spw_train / features")
    names = list(active.keys())
    aps = np.array([active[n][1] for n in names])
    w = aps / aps.sum()
    return [active[n][0] for n in names], w, dict(zip(names, aps.round(4))), names


def eval_at_threshold(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, proba),
        "ap": average_precision_score(y_true, proba),
        "acc": (pred == y_true).mean(),
        "prec": precision_score(y_true, pred, zero_division=0),
        "rec": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }

def print_model_report(name, model=None, probas=None):
    if probas is None:
        probas = _proba_splits(model)
    rows = [eval_split(_SPLIT_DATA[s]()[0], probas[s], s) for s in ("train", "val", "test")]
    tab = pd.DataFrame(rows)
    print(f"\n{name}")
    print(tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    t_rc, p_rc = best_recall_constrained_threshold(y_va, probas["val"])
    pred_te = (probas["test"] >= t_rc).astype(int)
    rk = recall_at_k(y_te, probas["test"], 0.10)
    print(f"  Val threshold (rec>={RECALL_FLOOR}): {t_rc:.4f}  val_prec={p_rc:.4f}")
    print(f"  Test @ threshold: prec={precision_score(y_te,pred_te,zero_division=0):.4f}  "
          f"rec={recall_score(y_te,pred_te,zero_division=0):.4f}  fleet={(pred_te==1).mean():.1%}")
    print(f"  Test @ top-10%: prec={rk['prec_at_k']:.4f}  rec={rk['recall_at_k']:.4f}  lift={rk['lift']:.2f}x")
    plot_roc_and_confusion(name, probas, threshold=0.5, op_threshold=t_rc)
    return tab, probas["test"]

In [ ]:
# CELL 9 — Logistic Regression baseline
lr = Pipeline([
    ("sc", StandardScaler()),
    ("clf", LogisticRegression(C=0.05, class_weight="balanced", max_iter=2000, solver="lbfgs", random_state=42)),
])
lr.fit(X_tr_fit, y_tr_fit)
lr_tab, _ = print_model_report("lr_improved", lr)

In [ ]:
# CELL 10 — LightGBM (prod-aligned; early stop on AUC; capped spw)
lgb = LGBMClassifier(
    n_estimators=800, learning_rate=0.05, num_leaves=63, max_depth=-1,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.5, scale_pos_weight=spw_train,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgb.fit(X_tr_fit, y_tr_fit, eval_set=[(X_va, y_va)], eval_metric="auc",
        callbacks=[early_stopping(50, verbose=False), log_evaluation(0)])
lgb_tab, lgb_proba = print_model_report("lgb_improved", lgb)

In [ ]:
# CELL 11 — XGBoost (prod-aligned)
xgb = XGBClassifier(
    n_estimators=800, learning_rate=0.05, max_depth=6, min_child_weight=20,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=spw_train, eval_metric="auc", random_state=42, n_jobs=-1,
    early_stopping_rounds=50,
)
xgb.fit(X_tr_fit, y_tr_fit, eval_set=[(X_va, y_va)], verbose=False)
xgb_tab, xgb_proba = print_model_report("xgb_improved", xgb)

In [ ]:
# CELL 12 — CatBoost (prod-aligned, depth 8)
cb = CatBoostClassifier(
    iterations=1000, learning_rate=0.05, depth=8, l2_leaf_reg=3,
    scale_pos_weight=spw_train, random_seed=42, verbose=0,
    eval_metric="AUC", od_type="Iter", od_wait=50,
)
cb.fit(X_tr_fit, y_tr_fit, eval_set=(X_va, y_va), use_best_model=True)
cb_tab, cb_proba = print_model_report("cb_improved", cb)

In [ ]:
# CELL 13 — Ensemble + deployment candidate (LR + boosters, prod-style isotonic)
_boosters_all = {"lgb": lgb, "xgb": xgb, "cb": cb}
_active_models, _ens_w, _va_aps, _ens_names = val_ap_weights_named(_boosters_all, X_va, y_va)
print(f"Booster ensemble val AP: {_va_aps}")
print(f"  weights={dict(zip(_ens_names, _ens_w.round(3)))}")

def _weighted_proba(models, weights, X):
    P = np.column_stack([m.predict_proba(X)[:, 1] for m in models])
    return P @ weights

p_tr_ens = _weighted_proba(_active_models, _ens_w, X_tr)
p_va_ens = _weighted_proba(_active_models, _ens_w, X_va)
p_te_ens = _weighted_proba(_active_models, _ens_w, X_te)

ens_probas = {"train": p_tr_ens, "val": p_va_ens, "test": p_te_ens}
ens_tab, _ = print_model_report("weighted_ensemble", probas=ens_probas)

# Deploy pool: LR + active boosters (val AP weights — LR often strongest on GATE)
_deploy_dict = {"lr": lr, **dict(zip(_ens_names, _active_models))}
_deploy_models, _deploy_w, _deploy_va_aps, _deploy_names = val_ap_weights_named(_deploy_dict, X_va, y_va)
print(f"Deploy val AP: {_deploy_va_aps}")
print(f"  deploy weights={dict(zip(_deploy_names, _deploy_w.round(3)))}")

p_va_deploy_raw = _weighted_proba(_deploy_models, _deploy_w, X_va)
p_te_deploy_raw = _weighted_proba(_deploy_models, _deploy_w, X_te)

class DeployAPEnsemble:
    """Val AP-weighted LR + LGB/XGB/CB + optional isotonic — SageMaker deploy artifact."""
    _estimator_type = "classifier"
    def __init__(self, models, weights, iso=None):
        self.models = models
        self.weights = np.asarray(weights)
        self.iso = iso
    def predict_proba(self, X):
        P = np.column_stack([m.predict_proba(X)[:, 1] for m in self.models])
        raw = (P * self.weights).sum(axis=1)
        if self.iso is not None:
            cal = self.iso.predict(raw)
            return np.column_stack([1 - cal, cal])
        return np.column_stack([1 - raw, raw])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

_deploy_iso = None
if USE_ISOTONIC:
    _cal_va, _cal_te, _deploy_iso, _iso_ok = calibrate_isotonic(p_va_deploy_raw, y_va, p_te_deploy_raw)
    _deploy_tag = "deploy_ap_isotonic_ensemble" if _iso_ok else "deploy_ap_ensemble (isotonic skipped)"
else:
    _cal_va, _cal_te, _deploy_iso, _iso_ok = p_va_deploy_raw, p_te_deploy_raw, None, False
    _deploy_tag = "deploy_ap_ensemble"

deploy_model = DeployAPEnsemble(_deploy_models, _deploy_w, _deploy_iso)
deploy_probas = {
    "train": deploy_model.predict_proba(X_tr)[:, 1],
    "val": _cal_va,
    "test": _cal_te,
}
deploy_tab, deploy_proba = print_model_report(_deploy_tag, probas=deploy_probas)
print(f"  [deploy] champion candidate: {_deploy_tag}  iso={'yes' if _iso_ok else 'no'}")

In [ ]:
# CELL 14 — Leaderboard (AP primary) + accuracy + deploy candidate
leaderboard = pd.concat([
    lr_tab.assign(model="lr_improved"),
    lgb_tab.assign(model="lgb_improved"),
    xgb_tab.assign(model="xgb_improved"),
    cb_tab.assign(model="cb_improved"),
    ens_tab.assign(model="weighted_ensemble"),
    deploy_tab.assign(model=_deploy_tag),
])
test_lb = leaderboard[leaderboard["split"] == "test"].sort_values("ap", ascending=False)

# Operational threshold from val (recall floor) — metrics @ tuned threshold
_op_t, _ = best_recall_constrained_threshold(y_va, deploy_probas["val"])
_op_row = eval_at_threshold(y_te, deploy_proba, _op_t)
_op_row["model"] = f"{_deploy_tag} @op_t"
_op_row["split"] = "test"

print("\n=== TEST LEADERBOARD (by AP, threshold=0.5) ===")
print(test_lb[["model", "auc", "ap", "acc", "prec", "rec", "f1"]].to_string(
    index=False, float_format=lambda x: f"{x:.4f}"))

print(f"\n=== DEPLOY CANDIDATE @ operational threshold (t={_op_t:.4f} from val) ===")
print(pd.DataFrame([_op_row])[["model", "auc", "ap", "acc", "prec", "rec", "f1"]].to_string(
    index=False, float_format=lambda x: f"{x:.4f}"))

prod = pd.DataFrame([
    {"model": "lgb_gate (prod+SMOTE)", "auc": 0.8078, "ap": 0.0947, "acc": np.nan},
    {"model": "cb_optuna (prod)",     "auc": 0.8213, "ap": 0.0897, "acc": np.nan},
])
print("\n=== Production reference ===")
print(prod.to_string(index=False))
best = test_lb.iloc[0]
_delta = best["ap"] - 0.0947
rk = recall_at_k(y_te, deploy_proba, 0.10)
print(f"\nBest @0.5: {best['model']}  AUC={best['auc']:.4f}  AP={best['ap']:.4f}  acc={best['acc']:.4f}  (delta vs prod: {_delta:+.4f})")
print(f"Deploy @ top-10%: prec={rk['prec_at_k']:.4f}  rec={rk['recall_at_k']:.4f}  lift={rk['lift']:.2f}x")

In [ ]:
# CELL 15 — All models: test-set ROC overlay (comparison)
_all_models = {
    "lr_improved": lr.predict_proba(X_te)[:, 1],
    "lgb_improved": lgb_proba,
    "xgb_improved": xgb_proba,
    "cb_improved": cb_proba,
    "weighted_ensemble": p_te_ens,
    _deploy_tag: deploy_proba,
}

fig, ax = plt.subplots(figsize=(8, 6))
for name, proba in _all_models.items():
    fpr, tpr, _ = roc_curve(y_te, proba)
    ax.plot(fpr, tpr, lw=2, label=f"{name}  AUC={roc_auc_score(y_te, proba):.3f}")
ax.plot([0, 1], [0, 1], "k--", alpha=0.35)
ax.set_title("TEST ROC — all models")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
path = os.path.join(OUT_DIR, "all_models_test_roc.png")
fig.savefig(path, dpi=120, bbox_inches="tight")
print(f"[plot] saved -> {path}")
plt.show()